# Appendix 5 — Findings

This notebook reads the full-corpus production codes (Appendix 3; committed as
`data/production_codes.json`, one record per decision) and reports the study's findings. It
produces, in order, the paper's **Table 1** (the corpus), the distributions and the share of late
claims, **Table 4** (gender/SOGIE allegations by claim timing — overall, per source, and for gender
and SOGIE separately), and **Table 5** (success rates by level, timing, and basis). Two things to
keep in mind throughout:

1. These are model codes, with measured error rates (Appendix 2). Most importantly, the model
   detects claims made more than one year after first entry with ≈0.95 precision and ≈0.80 recall
   against the benchmark labels — it under-counts late claims rather than inflating them.
2. The corpus is written, published decisions — not a probability sample of all refugee claims.
   Each source publishes selectively, so the analysis compares groups *within* a source or a
   level of the system and never pools sources whose selection runs in opposite directions
   (the published RPD decisions lean negative; the RLLR publishes only positive decisions).

The four coded questions (`timing`, `outcome`, `basis`, and the Federal-Court-only `jr_refugee` gate) and their allowed codes are set out in Appendix 1, with the full rubric in `data/golden_sample_coding_rubric.md`; the output legends below restate the code letters where they are used.

AI Disclosure: Generative AI tools were used in preparing these appendices, including for background research, data extraction, coding, drafting and editing (Claude Code, using Fable 5 and 5.5 and Opus 5 and 5.5).


In [1]:
from pathlib import Path
import json

import pandas as pd
from scipy.stats import fisher_exact

DATA = Path("data")   # every file these appendices need ships in this data/ folder

prod = pd.read_json(DATA / "production_codes.json")   # the full-corpus codes (Appendix 3)
print(f"{len(prod):,} coded decisions (2006-2025)")
prod.groupby("dataset").size()

32,462 coded decisions (2006-2025)


dataset
FC      11181
RAD     14156
RLLR      927
RPD      6198
dtype: int64

## The analysis universe

Federal Court cases first pass through the `jr_refugee` gate; only judicial reviews of RPD/RAD
refugee determinations stay in the analysis.

In [2]:
fc = prod[prod.dataset == "FC"]
print(f"FC gate: {(fc.jr_refugee == 'a').sum():,} of {len(fc):,} pre-filtered Federal Court cases are "
      f"judicial reviews of refugee determinations ({100 * (fc.jr_refugee == 'a').mean():.0f}%)")

coded = prod[((prod.dataset != "FC") | (prod.jr_refugee == "a")) & prod.timing.notna()].copy()
coded["gender_sogie"] = coded.basis.isin(["a", "b", "c"])
print(f"decisions in the analysis: {len(coded):,}")

FC gate: 7,571 of 11,181 pre-filtered Federal Court cases are judicial reviews of refugee determinations (68%)
decisions in the analysis: 28,852


### Table 1 — the corpus

Years covered, decisions categorized, and decisions in the analysis, per source. All 32,462
categorized decisions are dated 2006–2025; the RPD stops in 2020 (the Board stopped publishing
first-instance reasons), the RLLR runs 2019–2024, and the RAD begins in 2013 (its first year of
operation). Only Federal Court decisions passing the `jr_refugee` gate enter the analysis.


In [3]:
SOURCES = ["RPD", "RLLR", "RAD", "FC"]   # the paper's ordering: first instance, then appeal, then judicial review

prod["year"] = pd.to_datetime(prod.document_date, errors="coerce", utc=True).dt.year
table1 = pd.DataFrame({
    "years": prod.groupby("dataset").year.agg(lambda y: f"{y.min()}–{y.max()}"),
    "decisions categorized": prod.groupby("dataset").size(),
    "in the analysis": coded.groupby("dataset").size(),
}).loc[SOURCES]
table1.loc["Total"] = [f"{prod.year.min()}–{prod.year.max()}",
                       table1["decisions categorized"].sum(), table1["in the analysis"].sum()]
table1


,years,decisions categorized,in the analysis
dataset,,,
RPD,2006–2020,6198,6198
RLLR,2019–2024,927,927
RAD,2013–2025,14156,14156
FC,2006–2025,11181,7571
Total,2006–2025,32462,28852


## Distributions

In [4]:
print("timing (row %):  a = claim >1 year after first entry, b = within 1 year, c = unclear")
display((pd.crosstab(coded.dataset, coded.timing, margins=True, normalize="index") * 100).round(1))
print("basis (row %):  a = gender, b = SOGIE, c = both, d = other, e = unclear")
display((pd.crosstab(coded.dataset, coded.basis, margins=True, normalize="index") * 100).round(1))
print("counts:")
display(pd.crosstab(coded.dataset, coded.timing, margins=True))

timing (row %):  a = claim >1 year after first entry, b = within 1 year, c = unclear


timing,a,b,c
dataset,,,
FC,7.9,68.4,23.6
RAD,8.7,59.5,31.8
RLLR,9.5,47.8,42.7
RPD,8.6,73.5,18.0
All,8.5,64.5,27.0


basis (row %):  a = gender, b = SOGIE, c = both, d = other, e = unclear


basis,a,b,c,d,e
dataset,,,,,
FC,20.7,5.3,0.6,69.1,4.4
RAD,23.8,9.2,1.5,58.9,6.7
RLLR,21.4,30.5,3.5,44.7,0.0
RPD,23.7,4.8,0.6,68.2,2.7
All,22.9,7.9,1.1,63.1,5.0


counts:


timing,a,b,c,All
dataset,,,,
FC,600,5181,1790,7571
RAD,1236,8419,4501,14156
RLLR,88,443,396,927
RPD,531,4554,1113,6198
All,2455,18597,7800,28852


### How many claims are late

Counts and shares of the three timing codes per source, and — because “unclear” is a large
category — the share of late claims among decisions where the timing is clear (a or b). The
latter is the figure the paper reports (about one in nine).

Both shares are more likely under- than over-estimates. On the benchmark sample the model's late
share among clear-timing decisions (10.3%) runs below the benchmark's own (12.0%), because the
model misses more late claims than it invents (S8, from Appendix 2); and the decisions were
written before a claimant's *first* entry to Canada had legal consequences, so an earlier visit
that would now trigger the bar may simply go unmentioned and the claim be coded timely or unclear.


In [5]:
clear = coded[coded.timing.isin(["a", "b"])]
late_share = pd.DataFrame({
    "late (a)": clear.groupby("dataset").timing.apply(lambda t: (t == "a").sum()),
    "clear timing (a or b)": clear.groupby("dataset").size(),
}).loc[SOURCES]
late_share.loc["All sources"] = [(clear.timing == "a").sum(), len(clear)]
late_share["% late among clear-timing decisions"] = (100 * late_share["late (a)"] / late_share["clear timing (a or b)"]).round(1)
late_share


,late (a),clear timing (a or b),% late among clear-timing decisions
dataset,,,
RPD,531,5085,10.4
RLLR,88,531,16.6
RAD,1236,9655,12.8
FC,600,5781,10.4
All sources,2455,21052,11.7


In [6]:
print(f"timely share among clear-timing decisions: {(clear.timing == 'b').sum():,}/{len(clear):,} "
      f"= {100 * (clear.timing == 'b').mean():.1f}%")

timely share among clear-timing decisions: 18,597/21,052 = 88.3%


## The first research question: who claims late

Are claims made more than one year after first entry (timing = a) more likely to involve
gender and/or SOGIE allegations (basis = a, b, or c) than claims made within a year
(timing = b)? Decisions with unclear timing (c) are excluded from the comparison. For each
comparison we report the number of late claims, each timing group's gender/SOGIE share, the
odds ratio, and Fisher's exact test. Because the four sources differ in composition (the RLLR
publishes only positive decisions — see the known-answer check in the supplementary section;
the RAD and FC condition on an appeal or judicial review being taken), the comparison is repeated within each source; and because the headline
group pools gender-based and SOGIE-based allegations, it is also broken out (gender = a
gender-based allegation present, basis a or c; SOGIE = a SOGIE-based allegation present, basis
b or c; the small “both” group counts in each). This is the paper's **Table 4**.


In [7]:
def timing_association(sub, mask):
    """One row of Table 4: late vs timely claims in `sub`, on the binary outcome `mask`."""
    t = pd.crosstab(sub.timing, mask)
    orr, pv = fisher_exact([[t.loc["a", True], t.loc["a", False]],
                            [t.loc["b", True], t.loc["b", False]]])
    share = pd.crosstab(sub.timing, mask, normalize="index")[True] * 100
    return {"late claims (n)": t.loc["a"].sum(), "timely claims (n)": t.loc["b"].sum(),
            "% among late claims": round(share["a"], 1), "% among timely claims": round(share["b"], 1),
            "odds ratio": round(orr, 2), "Fisher p": pv}

rows = {"All sources": timing_association(coded, coded.gender_sogie)}
for d in SOURCES:
    sub = coded[coded.dataset == d]
    rows[d] = timing_association(sub, sub.gender_sogie)
rows["All sources: gender-based allegation present (a or c)"] = timing_association(coded, coded.basis.isin(["a", "c"]))
rows["All sources: SOGIE allegation present (b or c)"] = timing_association(coded, coded.basis.isin(["b", "c"]))

table4 = pd.DataFrame(rows).T
for col in ["late claims (n)", "timely claims (n)"]:
    table4[col] = table4[col].astype(int)
table4["Fisher p"] = table4["Fisher p"].map(lambda p: "< 0.001" if p < 0.001 else f"{p:.2f}")
display(table4)

unclear = coded[coded.timing == "c"]
print(f"excluded from the comparison: {len(unclear):,} decisions with unclear timing "
      f"({100 * unclear.gender_sogie.mean():.1f}% gender/SOGIE)")


,late claims (n),timely claims (n),% among late claims,% among timely claims,odds ratio,Fisher p
All sources,2455,18597,41.5,31.1,1.58,< 0.001
RPD,531,4554,41.4,28.1,1.81,< 0.001
RLLR,88,443,61.4,52.4,1.44,0.13
RAD,1236,8419,41.7,35.0,1.33,< 0.001
FC,600,5181,38.3,25.5,1.81,< 0.001
All sources: gender-based allegation present (a or c),2455,18597,29.6,23.9,1.34,< 0.001
All sources: SOGIE allegation present (b or c),2455,18597,13.4,8.2,1.73,< 0.001


excluded from the comparison: 7,800 decisions with unclear timing (30.8% gender/SOGIE)


In [8]:
# the unrounded Fisher p-values behind Table 4 (the table prints "< 0.001" below that threshold)
print("Table 4, exact Fisher p:")
print(pd.Series({k: f"{v['Fisher p']:.3g}" for k, v in rows.items()}).to_string())

Table 4, exact Fisher p:
All sources                                              1.17e-24
RPD                                                      5.13e-10
RLLR                                                        0.129
RAD                                                      4.04e-06
FC                                                       1.04e-10
All sources: gender-based allegation present (a or c)    1.52e-09
All sources: SOGIE allegation present (b or c)           7.15e-16


## The second research question: do late claims succeed, and are errors against them caught?

A one-year bar is premised on late claims being less deserving. The outcome codes speak to this
at each level of the system — first instance (the published RPD decisions and the RLLR), the
refugee appeal (RAD), and judicial review (FC) — always from the principal claimant's
perspective, restricted to decisions with a clear successful/unsuccessful outcome. This is the
paper's **Table 5**.

Outcome rates are strongly shaped by which decisions get written and published, so groups are
compared *within* a level, never across levels. At first instance the two sources select for
outcomes in opposite directions — the published RPD decisions lean heavily negative, while every
RLLR decision is positive — so the RPD and RLLR are reported separately, and the first-instance
record is used only for the count of late claims the Board found well-founded (a floor, not a
rate to be compared). Odds ratios are therefore computed only at the RAD and the Federal Court.


In [9]:
ok = coded[coded.outcome.isin(["a", "b"])].copy()
ok["gender"] = ok.basis.isin(["a", "c"])
ok["sogie"] = ok.basis.isin(["b", "c"])

LEVELS = [("RPD (first instance, published)", ok.dataset == "RPD"),
          ("RLLR (first instance, published positive)", ok.dataset == "RLLR"),
          ("RAD appeal", ok.dataset == "RAD"),
          ("FC judicial review", ok.dataset == "FC")]
COMPARED = ["RAD appeal", "FC judicial review"]   # the only levels where a within-level comparison is meaningful

def rate(sub):
    return f"{(sub.outcome == 'a').sum()}/{len(sub)} = {(sub.outcome == 'a').mean() * 100:.0f}%" if len(sub) else "—"

rows = []
for level, lmask in LEVELS:
    sub = ok[lmask]
    for timing, tname in [("a", "late (>1 yr)"), ("b", "timely (≤1 yr)")]:
        s = sub[sub.timing == timing]
        rows.append({"level": level, "claims": tname, "all": rate(s),
                     "gender/SOGIE": rate(s[s.gender_sogie]),
                     "gender-based present (a+c)": rate(s[s.gender]), "SOGIE present (b+c)": rate(s[s.sogie]),
                     "other (d/e)": rate(s[~s.gender_sogie])})
table5 = pd.DataFrame(rows).set_index(["level", "claims"])
display(table5)

print("late claims: success of gender/SOGIE claims vs other claims, within each compared level:")
for level, lmask in LEVELS:
    if level not in COMPARED:
        continue
    s = ok[lmask & (ok.timing == "a")]
    t = pd.crosstab(s.gender_sogie, s.outcome == "a")
    orr, pv = fisher_exact([[t.loc[True, True], t.loc[True, False]],
                            [t.loc[False, True], t.loc[False, False]]])
    print(f"  {level}: odds ratio {orr:.2f}, Fisher p {pv:.2g}")


all  \
level                                     claims                            
RPD (first instance, published)           late (>1 yr)        10/531 = 2%   
                                          timely (≤1 yr)    210/4549 = 5%   
RLLR (first instance, published positive) late (>1 yr)       88/88 = 100%   
                                          timely (≤1 yr)   442/443 = 100%   
RAD appeal                                late (>1 yr)     282/1236 = 23%   
                                          timely (≤1 yr)  2319/8418 = 28%   
FC judicial review                        late (>1 yr)      187/598 = 31%   
                                          timely (≤1 yr)  1846/5175 = 36%   

                                                            gender/SOGIE  \
level                                     claims                           
RPD (first instance, published)           late (>1 yr)        3/220 = 1%   
                                          timely (≤1 yr)    77/1278 = 6%   
RLLR (first instance, published positive) late (>1 yr)      54/54 = 100%   
                                          timely (≤1 yr)  231/232 = 100%   
RAD appeal                                late (>1 yr)     143/516 = 28%   
                                          timely (≤1 yr)  925/2943 = 31%   
FC judicial review                        late (>1 yr)      88/230 = 38%   
                                          timely (≤1 yr)  481/1323 = 36%   

                                                         gender-based present (a+c)  \
level                                     claims                                      
RPD (first instance, published)           late (>1 yr)                   3/177 = 2%   
                                          timely (≤1 yr)               64/1071 = 6%   
RLLR (first instance, published positive) late (>1 yr)                 18/18 = 100%   
                                          timely (≤1 yr)              116/117 = 99%   
RAD appeal                                late (>1 yr)                 94/338 = 28%   
                                          timely (≤1 yr)             650/2179 = 30%   
FC judicial review                        late (>1 yr)                 71/194 = 37%   
                                          timely (≤1 yr)             397/1080 = 37%   

                                                         SOGIE present (b+c)  \
level                                     claims                               
RPD (first instance, published)           late (>1 yr)             1/50 = 2%   
                                          timely (≤1 yr)         17/235 = 7%   
RLLR (first instance, published positive) late (>1 yr)          38/38 = 100%   
                                          timely (≤1 yr)      132/132 = 100%   
RAD appeal                                late (>1 yr)          54/199 = 27%   
                                          timely (≤1 yr)       321/889 = 36%   
FC judicial review                        late (>1 yr)           21/43 = 49%   
                                          timely (≤1 yr)        94/275 = 34%   

                                                              other (d/e)  
level                                     claims                           
RPD (first instance, published)           late (>1 yr)         7/311 = 2%  
                                          timely (≤1 yr)    133/3271 = 4%  
RLLR (first instance, published positive) late (>1 yr)       34/34 = 100%  
                                          timely (≤1 yr)   211/211 = 100%  
RAD appeal                                late (>1 yr)      139/720 = 19%  
                                          timely (≤1 yr)  1394/5475 = 25%  
FC judicial review                        late (>1 yr)       99/368 = 27%  
                                          timely (≤1 yr)  1365/3852 = 35%

late claims: success of gender/SOGIE claims vs other claims, within each compared level:
  RAD appeal: odds ratio 1.60, Fisher p 0.00058
  FC judicial review: odds ratio 1.68, Fisher p 0.0049


**Why first instance is not compared.** Pooling the RPD and the RLLR would produce a
first-instance “finding” that is an artefact of composition. The RLLR skews late and skews toward
gender and SOGIE claims (Table 4), and every RLLR decision is a success; so if the two
were pooled, late gender/SOGIE claims would appear to succeed far more often than other late
claims. Within the published RPD decisions alone the gap disappears, as the cell below shows.
The study therefore uses the first-instance record only as a floor — the number of late claims
the Board found well-founded — and draws its outcome comparisons from the RAD and the Federal
Court.


In [10]:
rpd_late = ok[(ok.dataset == "RPD") & (ok.timing == "a")]
t = pd.crosstab(rpd_late.gender_sogie, rpd_late.outcome == "a")
orr, pv = fisher_exact([[t.loc[True, True], t.loc[True, False]],
                        [t.loc[False, True], t.loc[False, False]]])
print(f"RPD only, late claims: gender/SOGIE {rate(rpd_late[rpd_late.gender_sogie])} vs "
      f"other {rate(rpd_late[~rpd_late.gender_sogie])}   (odds ratio {orr:.2f}, Fisher p {pv:.2f})")
rllr_late = ok[(ok.dataset == "RLLR") & (ok.timing == "a")]
print(f"RLLR only, late claims: {rate(rllr_late)} — every RLLR decision is positive, so no comparison is possible")
fi_late = ok[ok.dataset.isin(["RPD", "RLLR"]) & (ok.timing == "a")]
print(f"floor: the Board granted protection in {(fi_late.outcome == 'a').sum()} published late claims, "
      f"{(fi_late[fi_late.gender_sogie].outcome == 'a').sum()} of them involving gender or SOGIE allegations")


RPD only, late claims: gender/SOGIE 3/220 = 1% vs other 7/311 = 2%   (odds ratio 0.60, Fisher p 0.53)
RLLR only, late claims: 88/88 = 100% — every RLLR decision is positive, so no comparison is possible
floor: the Board granted protection in 98 published late claims, 57 of them involving gender or SOGIE allegations


**First instance (RPD and RLLR, reported separately).** The published RPD decisions lean
heavily negative and the RLLR is exclusively positive, so first-instance success rates track the
mix of sources rather than how adjudicators treated the claims, and are not compared. What the
first-instance record establishes is a floor: across the two sources the Board granted
protection in 98 published late claims (57 of them involving gender or SOGIE allegations) —
claims made more than a year after entry that the Board itself found well-founded, and that a
one-year bar would have kept from the Board entirely.

**RAD.** The refugee appeal exists to catch first-instance error, and it is catching errors
against late claims: 282 of 1,236 late-claim RAD appeals with a clear outcome ended in the
claimant's favour (23%, against 28% for timely claims). The catch rate is higher for
gender/SOGIE late claims (28% vs 19%; odds ratio 1.60, p ≈ 0.0006), with gender and SOGIE
behaving alike (28% and 27%). Under Bill C-12, late claims would be diverted to the Pre-Removal
Risk Assessment, a process with no equivalent of this appeal. (Outcome is coded from the
claimant's perspective, so a small share of these successes are dismissals of Minister's
appeals; most RAD appeals are claimant appeals.)

**Federal Court.** Errors against late claims are being caught at judicial review too: 187 of
598 late-claim JRs with a clear outcome were granted (31%, against 36% for timely claims). The
gender/SOGIE difference persists among late claims (38% vs 27%, odds ratio 1.68, p ≈ 0.005) —
despite judicial review being deferential precisely on credibility, the issue at the heart of
most gender/SOGIE claims. Among *timely* claims, FC success rates are flat across claim types
(36% vs 35%; odds ratio 1.04, p ≈ 0.55): the gender/SOGIE gap appears only among late claims,
and the difference between the late and timely odds ratios is itself statistically supported
(ratio of odds ratios 1.62, p ≈ 0.012; supplementary table S4).

**Two patterns cut the other way.** Late claims succeed less often than timely claims at both
the RAD (23% vs 28%) and the Federal Court (31% vs 36%), consistent with late claims being, on
average, somewhat weaker. Supplementary table S5 shows where that deficit sits: late claims
without gender/SOGIE allegations do clearly worse than timely ones (RAD 19% vs 25%, FC 27% vs
35%, both p < 0.001), whereas late gender/SOGIE claims succeed about as often as timely ones
(RAD 28% vs 31%, p ≈ 0.10; FC 38% vs 36%, p ≈ 0.60) — with one exception: at the RAD, late
SOGIE claims succeed less often than timely SOGIE claims (27% vs 36%, p ≈ 0.02). And at the
RAD the gender/SOGIE advantage is not confined to late claims (timely: 31% vs 25%, odds ratio
1.34): the gap among late claims is numerically wider (28% vs 19%, odds ratio 1.60), but the
two odds ratios are not statistically distinguishable (ratio 1.19, p ≈ 0.22; S4). The Federal
Court shows the sharpest version of the pattern — no gap among timely claims, an eleven-point
gap among late ones — and there the late/timely difference is supported (p ≈ 0.012).


## Supplementary analyses

The cells below add checks that the discussion relies on but that the tables above do not
print. They are additive: Tables 1, 4, and 5 above are unchanged. Throughout, “other or unclear
basis (d/e)” is the complement of the gender/SOGIE group — it includes the decisions whose
basis the model could not determine (code `e`), which Tables 4 and 5 count as “other”; S6
re-runs the main comparisons with those decisions dropped.

### S1. RLLR known-answer check

The RLLR publishes only positive decisions, so its outcome codes are a known-answer test of the
outcome question on 927 decisions — including the 396 with unclear timing that Table 5 leaves
out. Every decision not coded successful is listed.

In [11]:
rllr = coded[coded.dataset == "RLLR"]
print("RLLR outcome codes (a = successful, b = unsuccessful, c = unclear) by timing (a = late, b = timely, c = unclear):")
display(pd.crosstab(rllr.timing.rename("timing"), rllr.outcome.rename("outcome"), margins=True))
print(f"coded successful: {(rllr.outcome == 'a').sum()} of {len(rllr)} = {100 * (rllr.outcome == 'a').mean():.1f}%")
print("not coded successful:")
display(rllr[rllr.outcome != "a"][["uid", "citation", "document_date", "timing", "outcome", "basis"]])

RLLR outcome codes (a = successful, b = unsuccessful, c = unclear) by timing (a = late, b = timely, c = unclear):


outcome,a,b,All
timing,,,
a,88,0,88
b,442,1,443
c,393,3,396
All,923,4,927


coded successful: 923 of 927 = 99.6%
not coded successful:


,uid,citation,document_date,timing,outcome,basis
25695,RLLR_00358,2019 RLLR 204,2019-04-11,c,b,a
25704,RLLR_00367,2019 RLLR 214,2019-06-19,b,b,a
25706,RLLR_00369,2019 RLLR 216,2019-10-02,c,b,d
25708,RLLR_00371,2019 RLLR 218,2019-10-22,c,b,d


Four of the 927 are coded unsuccessful: three with unclear timing and the one timely decision
behind Table 5's 442/443. Read individually, none is a model error: all four are split
decisions in which the principal claimant was refused while an associated claimant was
accepted, so the outcome code — taken from the principal claimant's perspective — is correct.
In 2019 RLLR 204 the principal claimant (the mother) and her son were refused and her daughter
accepted; in 2019 RLLR 214 the first-named claimant was refused and her adult daughter and three
grandchildren accepted; in 2019 RLLR 216 the father and the youngest child were excluded under
Article 1E (permanent residence in Japan) while the mother and two older children were
accepted; in 2019 RLLR 218 the principal claimant (the mother) was refused and her minor son
accepted. Each published RLLR decision therefore does contain a grant of protection, which is
the sense in which the RLLR publishes only positive decisions; 923 of 927 principal claimants
succeeded.

### S2. The association read the other way

Table 4 reads the association as the gender/SOGIE share of late vs timely claims. Read the
other way — the late share among gender/SOGIE claims vs among other claims, on clear-timing
decisions — with the risk ratio, for the pooled group and for each component.

In [12]:
clear = coded[coded.timing.isin(["a", "b"])]

def late_rate(mask, name):
    g, o = clear[mask], clear[~mask]
    lg, lo = (g.timing == "a").sum(), (o.timing == "a").sum()
    return {"group": name, "late among the group": f"{lg:,}/{len(g):,} = {100 * lg / len(g):.1f}%",
            "late among all other claims": f"{lo:,}/{len(o):,} = {100 * lo / len(o):.1f}%",
            "risk ratio": round((lg / len(g)) / (lo / len(o)), 2)}

display(pd.DataFrame([late_rate(clear.gender_sogie, "gender/SOGIE allegation (a, b or c)"),
                      late_rate(clear.basis.isin(["a", "c"]), "gender-based allegation present (a or c)"),
                      late_rate(clear.basis.isin(["b", "c"]), "SOGIE allegation present (b or c)")]).set_index("group"))

,late among the group,late among all other claims,risk ratio
group,,,
"gender/SOGIE allegation (a, b or c)","1,020/6,797 = 15.0%","1,435/14,255 = 10.1%",1.49
gender-based allegation present (a or c),"727/5,175 = 14.0%","1,728/15,877 = 10.9%",1.29
SOGIE allegation present (b or c),"330/1,861 = 17.7%","2,125/19,191 = 11.1%",1.60


### S3. Gender-based and SOGIE allegations within each source

Table 4 breaks the headline group into its two components only with all sources pooled. The
same `timing_association` rows, for each component within each source.

In [13]:
rows_s3 = {}
for d in SOURCES:
    sub = coded[coded.dataset == d]
    rows_s3[f"{d}: gender-based allegation present (a or c)"] = timing_association(sub, sub.basis.isin(["a", "c"]))
    rows_s3[f"{d}: SOGIE allegation present (b or c)"] = timing_association(sub, sub.basis.isin(["b", "c"]))
s3 = pd.DataFrame(rows_s3).T
for col in ["late claims (n)", "timely claims (n)"]:
    s3[col] = s3[col].astype(int)
s3["Fisher p"] = s3["Fisher p"].map(lambda p: f"{p:.2g}")
display(s3)

,late claims (n),timely claims (n),% among late claims,% among timely claims,odds ratio,Fisher p
RPD: gender-based allegation present (a or c),531,4554,33.3,23.5,1.62,1.5e-06
RPD: SOGIE allegation present (b or c),531,4554,9.4,5.2,1.91,0.00019
RLLR: gender-based allegation present (a or c),88,443,20.5,26.4,0.72,0.28
RLLR: SOGIE allegation present (b or c),88,443,43.2,29.8,1.79,0.017
RAD: gender-based allegation present (a or c),1236,8419,27.3,25.9,1.08,0.28
RAD: SOGIE allegation present (b or c),1236,8419,16.1,10.6,1.63,3.6e-08
FC: gender-based allegation present (a or c),600,5181,32.3,20.8,1.81,6.6e-10
FC: SOGIE allegation present (b or c),600,5181,7.2,5.3,1.38,0.071


### S4. Is the late-claim gap different from the timely-claim gap?

Table 5's discussion contrasts the gender/SOGIE-vs-other success gap among late claims with the
gap among timely claims at the RAD and the Federal Court. The table gives both odds ratios with
Fisher's test; the lines beneath test whether the two differ, with a Wald z-test on the
difference of the log odds ratios (Woolf standard errors).

In [14]:
import numpy as np
from scipy.stats import norm

def or_table(sub):
    """2x2 for gender/SOGIE vs other claims against success: [[gs ok, gs not], [other ok, other not]]."""
    t = pd.crosstab(sub.gender_sogie, sub.outcome == "a")
    return [[t.loc[True, True], t.loc[True, False]], [t.loc[False, True], t.loc[False, False]]]

def log_or(tab):
    (a, b), (c, d) = tab
    return np.log((a * d) / (b * c)), np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)   # Woolf standard error

rows_s4, tests = [], []
for level, lmask in LEVELS:
    if level not in COMPARED:
        continue
    lo = {}
    for timing, tname in [("a", "late (>1 yr)"), ("b", "timely (≤1 yr)")]:
        tab = or_table(ok[lmask & (ok.timing == timing)])
        orr, pv = fisher_exact(tab)
        lo[timing] = log_or(tab)
        rows_s4.append({"level": level, "claims": tname,
                        "gender/SOGIE": f"{tab[0][0]}/{sum(tab[0])} = {100 * tab[0][0] / sum(tab[0]):.0f}%",
                        "other or unclear basis (d/e)": f"{tab[1][0]}/{sum(tab[1])} = {100 * tab[1][0] / sum(tab[1]):.0f}%",
                        "odds ratio": round(orr, 2), "Fisher p": f"{pv:.2g}"})
    diff = lo["a"][0] - lo["b"][0]
    z = diff / np.sqrt(lo["a"][1] ** 2 + lo["b"][1] ** 2)
    tests.append(f"  {level}: ratio of odds ratios (late / timely) {np.exp(diff):.2f}, Wald z {z:.2f}, p {2 * norm.sf(abs(z)):.3f}")
display(pd.DataFrame(rows_s4).set_index(["level", "claims"]))
print("do the late and timely odds ratios differ?")
print("\n".join(tests))

gender/SOGIE  \
level              claims                           
RAD appeal         late (>1 yr)     143/516 = 28%   
                   timely (≤1 yr)  925/2943 = 31%   
FC judicial review late (>1 yr)      88/230 = 38%   
                   timely (≤1 yr)  481/1323 = 36%   

                                  other or unclear basis (d/e)  odds ratio  \
level              claims                                                    
RAD appeal         late (>1 yr)                  139/720 = 19%        1.60   
                   timely (≤1 yr)              1394/5475 = 25%        1.34   
FC judicial review late (>1 yr)                   99/368 = 27%        1.68   
                   timely (≤1 yr)              1365/3852 = 35%        1.04   

                                  Fisher p  
level              claims                   
RAD appeal         late (>1 yr)    0.00058  
                   timely (≤1 yr)  6.2e-09  
FC judicial review late (>1 yr)     0.0049  
                   timely (≤1 yr)     0.55

do the late and timely odds ratios differ?
  RAD appeal: ratio of odds ratios (late / timely) 1.19, Wald z 1.22, p 0.222
  FC judicial review: ratio of odds ratios (late / timely) 1.62, Wald z 2.51, p 0.012


### S5. Late vs timely success within each basis group

Table 5 compares claim types within a timing group; this table compares timing groups within
each claim type, at the two compared levels, with the odds ratio (late vs timely) and Fisher's
test.

In [15]:
GROUPS = [("all", lambda s: pd.Series(True, index=s.index)),
          ("gender/SOGIE (a, b or c)", lambda s: s.gender_sogie),
          ("gender-based present (a or c)", lambda s: s.gender),
          ("SOGIE present (b or c)", lambda s: s.sogie),
          ("other or unclear basis (d/e)", lambda s: ~s.gender_sogie)]
rows_s5 = []
for level, lmask in LEVELS:
    if level not in COMPARED:
        continue
    for gname, gmask in GROUPS:
        s = ok[lmask]
        s = s[gmask(s)]
        t = pd.crosstab(s.timing, s.outcome == "a")
        orr, pv = fisher_exact([[t.loc["a", True], t.loc["a", False]], [t.loc["b", True], t.loc["b", False]]])
        rows_s5.append({"level": level, "basis group": gname, "late (>1 yr)": rate(s[s.timing == "a"]),
                        "timely (≤1 yr)": rate(s[s.timing == "b"]),
                        "odds ratio (late vs timely)": round(orr, 2), "Fisher p": f"{pv:.2g}"})
display(pd.DataFrame(rows_s5).set_index(["level", "basis group"]))

late (>1 yr)  \
level              basis group                                     
RAD appeal         all                            282/1236 = 23%   
                   gender/SOGIE (a, b or c)        143/516 = 28%   
                   gender-based present (a or c)    94/338 = 28%   
                   SOGIE present (b or c)           54/199 = 27%   
                   other or unclear basis (d/e)    139/720 = 19%   
FC judicial review all                             187/598 = 31%   
                   gender/SOGIE (a, b or c)         88/230 = 38%   
                   gender-based present (a or c)    71/194 = 37%   
                   SOGIE present (b or c)            21/43 = 49%   
                   other or unclear basis (d/e)     99/368 = 27%   

                                                   timely (≤1 yr)  \
level              basis group                                      
RAD appeal         all                            2319/8418 = 28%   
                   gender/SOGIE (a, b or c)        925/2943 = 31%   
                   gender-based present (a or c)   650/2179 = 30%   
                   SOGIE present (b or c)           321/889 = 36%   
                   other or unclear basis (d/e)   1394/5475 = 25%   
FC judicial review all                            1846/5175 = 36%   
                   gender/SOGIE (a, b or c)        481/1323 = 36%   
                   gender-based present (a or c)   397/1080 = 37%   
                   SOGIE present (b or c)            94/275 = 34%   
                   other or unclear basis (d/e)   1365/3852 = 35%   

                                                  odds ratio (late vs timely)  \
level              basis group                                                  
RAD appeal         all                                                   0.78   
                   gender/SOGIE (a, b or c)                              0.84   
                   gender-based present (a or c)                         0.91   
                   SOGIE present (b or c)                                0.66   
                   other or unclear basis (d/e)                          0.70   
FC judicial review all                                                   0.82   
                   gender/SOGIE (a, b or c)                              1.08   
                   gender-based present (a or c)                         0.99   
                   SOGIE present (b or c)                                1.84   
                   other or unclear basis (d/e)                          0.67   

                                                 Fisher p  
level              basis group                             
RAD appeal         all                             0.0004  
                   gender/SOGIE (a, b or c)         0.098  
                   gender-based present (a or c)     0.48  
                   SOGIE present (b or c)           0.017  
                   other or unclear basis (d/e)   0.00028  
FC judicial review all                              0.034  
                   gender/SOGIE (a, b or c)           0.6  
                   gender-based present (a or c)        1  
                   SOGIE present (b or c)           0.087  
                   other or unclear basis (d/e)   0.00088

### S6. Robustness: decisions with unclear basis

`gender_sogie` is `basis in {a, b, c}`, so the decisions whose basis the model could not
determine (`e`, 5% of the analysis set) count as “other” in Tables 4 and 5, whereas decisions
with unclear *timing* are excluded from the comparisons. The cell below re-runs Table 4 and
the two late-claim outcome odds ratios with the `e` decisions dropped.

In [16]:
no_e = coded[coded.basis != "e"]
rows_s6 = {"All sources": timing_association(no_e, no_e.gender_sogie)}
for d in SOURCES:
    sub = no_e[no_e.dataset == d]
    rows_s6[d] = timing_association(sub, sub.gender_sogie)
rows_s6["All sources: gender-based allegation present (a or c)"] = timing_association(no_e, no_e.basis.isin(["a", "c"]))
rows_s6["All sources: SOGIE allegation present (b or c)"] = timing_association(no_e, no_e.basis.isin(["b", "c"]))
s6 = pd.DataFrame(rows_s6).T
for col in ["late claims (n)", "timely claims (n)"]:
    s6[col] = s6[col].astype(int)
s6["Fisher p"] = s6["Fisher p"].map(lambda p: f"{p:.2g}")
print("Table 4 with unclear-basis decisions dropped:")
display(s6)

print("late claims: success of gender/SOGIE claims vs other claims, unclear-basis decisions dropped:")
for level, lmask in LEVELS:
    if level not in COMPARED:
        continue
    orr, pv = fisher_exact(or_table(ok[lmask & (ok.timing == "a") & (ok.basis != "e")]))
    print(f"  {level}: odds ratio {orr:.2f}, Fisher p {pv:.2g}")

Table 4 with unclear-basis decisions dropped:


,late claims (n),timely claims (n),% among late claims,% among timely claims,odds ratio,Fisher p
All sources,2439,18326,41.8,31.5,1.56,1.6e-23
RPD,528,4497,41.7,28.4,1.80,9.7e-10
RLLR,88,443,61.4,52.4,1.44,0.13
RAD,1233,8315,41.8,35.4,1.31,1.3e-05
FC,590,5071,39.0,26.1,1.81,1.3e-10
All sources: gender-based allegation present (a or c),2439,18326,29.8,24.3,1.32,5.3e-09
All sources: SOGIE allegation present (b or c),2439,18326,13.5,8.4,1.72,1.6e-15


late claims: success of gender/SOGIE claims vs other claims, unclear-basis decisions dropped:
  RAD appeal: odds ratio 1.59, Fisher p 0.00074
  FC judicial review: odds ratio 1.67, Fisher p 0.0048


### S7. Robustness: who has unclear timing

Reading 4 below warns that the clear-timing subset may be unrepresentative if redaction
correlates with claim type. The share of decisions with unclear timing by source and claim
type, and the gender/SOGIE share of the unclear-timing group against the late and timely groups.

In [17]:
unc = coded.assign(unclear=coded.timing == "c",
                   group=coded.gender_sogie.map({True: "gender/SOGIE", False: "other or unclear basis (d/e)"}))

print("% of decisions with unclear timing, by source and claim type:")
display((pd.crosstab(unc.dataset, unc.group, values=unc.unclear, aggfunc="mean") * 100).round(1).reindex(SOURCES))
print(f"pooled: gender/SOGIE share {100 * unc[unc.unclear].gender_sogie.mean():.1f}% in the unclear-timing group "
      f"vs {100 * unc[unc.timing == 'b'].gender_sogie.mean():.1f}% among timely claims "
      f"and {100 * unc[unc.timing == 'a'].gender_sogie.mean():.1f}% among late claims")

% of decisions with unclear timing, by source and claim type:


group,gender/SOGIE,other or unclear basis (d/e)
dataset,,
RPD,16.8,18.4
RLLR,44.2,40.8
RAD,29.0,33.3
FC,22.8,24.0


pooled: gender/SOGIE share 30.8% in the unclear-timing group vs 31.1% among timely claims and 41.5% among late claims


### S8. Figures from Appendices 2 and 4 cited in this notebook

“How many claims are late” and “Reading the findings” cite the model's late-claim precision and
recall and its late share on the benchmark sample (Appendix 2), and the human review's tallies
(Appendix 4). They are recomputed here from the committed files so that every figure has a
visible source in this notebook.

In [18]:
val = pd.read_json(DATA / "golden_predictions_qwen.json")
s = val[val.gold_timing.notna()]
tp = ((s.gold_timing == "a") & (s.pred_timing == "a")).sum()
fp = ((s.gold_timing != "a") & (s.pred_timing == "a")).sum()
fn = ((s.gold_timing == "a") & (s.pred_timing != "a")).sum()
print(f"Appendix 2 - late-claim codes against the benchmark labels: "
      f"precision {tp}/{tp + fp} = {tp / (tp + fp):.3f}, recall {tp}/{tp + fn} = {tp / (tp + fn):.3f}")
ref_clear, mod_clear = s[s.gold_timing.isin(["a", "b"])], s[s.pred_timing.isin(["a", "b"])]
print(f"Appendix 2 - late share among clear-timing benchmark decisions: benchmark labels "
      f"{(ref_clear.gold_timing == 'a').sum()}/{len(ref_clear)} = {100 * (ref_clear.gold_timing == 'a').mean():.1f}%, "
      f"model {(mod_clear.pred_timing == 'a').sum()}/{len(mod_clear)} = {100 * (mod_clear.pred_timing == 'a').mean():.1f}%")

YES = "Yes: the model's code is reasonable"
NA = "Not applicable (this question has no code for this case)"
def review_counts(name):
    labels = json.loads((DATA / name).read_text(encoding="utf-8"))["labels"]
    answers = [a for l in labels for a in l["answers"].values() if a != NA]
    late = [l["answers"]["timing_reasonable"] for l in labels if l["doc_meta"]["qwen_timing"] == "a"]
    return sum(a == YES for a in answers), len(answers), sum(a == YES for a in late), len(late)
r100, l25 = review_counts("reasonableness_review.json"), review_counts("reasonableness_review_late25.json")
print(f"Appendix 4 - random 100: {r100[0]}/{r100[1]} codes judged reasonable; "
      f"late-claim codes upheld: {r100[2]}/{r100[3]} (random 100) + {l25[2]}/{l25[3]} (targeted) "
      f"= {r100[2] + l25[2]}/{r100[3] + l25[3]}")

Appendix 2 - late-claim codes against the benchmark labels: precision 37/39 = 0.949, recall 37/46 = 0.804
Appendix 2 - late share among clear-timing benchmark decisions: benchmark labels 46/383 = 12.0%, model 39/379 = 10.3%
Appendix 4 - random 100: 292/293 codes judged reasonable; late-claim codes upheld: 8/8 (random 100) + 24/25 (targeted) = 32/33


## Reading the findings

1. **The association the advocates predicted is present.** Claims coded as made more than one
   year after first entry are markedly more likely to involve gender and/or SOGIE allegations
   (about 42% vs 31% for timely claims; odds ratio ≈1.6, p ≈ 10⁻²⁴). The direction holds within
   every source (odds ratios 1.3–1.8; statistically significant everywhere except the small
   RLLR corpus), and, pooled across sources, for gender-based and SOGIE-based allegations
   separately (odds ratios ≈1.3 and ≈1.7). Within sources the two components are less uniform
   (S3): SOGIE allegations are more common among late claims in all four sources (odds ratios
   1.4–1.9; p ≈ 0.07 at the Federal Court, below 0.02 elsewhere), while for gender-based
   allegations the association is strong at the RPD and the Federal Court (odds ratios 1.6 and
   1.8), absent at the RAD (1.08, p ≈ 0.28), and reversed but not significant in the small RLLR
   corpus (0.72, p ≈ 0.28).
2. **Late claims include claims adjudicators found well-founded — and errors against late
   claims are being caught on appeal and review.** The Board granted protection in 98 published
   late claims; at the RAD, 23% of late-claim appeals ended in the claimant's favour; at the
   Federal Court, 31% of late-claim JRs were granted. At both the RAD and the Federal Court —
   the levels where a within-level comparison is meaningful — late gender/SOGIE claims succeeded
   more often than other late claims. Under a one-year bar these claims would not have reached
   the Board — nor the RAD appeal that is catching first-instance errors.
3. **Measurement error most likely understates the association.** Against the benchmark labels
   the model's late-claim group has ≈0.95 precision / ≈0.80 recall, and its timing errors lean
   conservative (mostly “unclear” rather than wrong-direction). Classification noise of this
   kind tends to dilute a real association rather than manufacture one.
4. About a quarter of cases have unclear timing, largely because published decisions redact
   dates. If redaction practices correlate with claim type, the clear-timing subset may not be
   fully representative — a reason the “unclear” group is reported throughout rather than
   silently dropped. S7 shows the unclear share differs little by claim type within each source
   (1–4 points), and that the unclear group's gender/SOGIE share (30.8%) is essentially the timely
   group's (31.1%) — marginally below it, and well below the late group's (41.5%). S6 shows that
   dropping the decisions with unclear *basis* leaves every Table 4 odds ratio within 0.02 of
   its value and the late-claim outcome odds ratios at 1.59 (RAD) and 1.67 (FC).
5. **Scope.** The corpus reflects the August 2026 widening of the FC pre-filter (Appendix 3),
   and the benchmark labels are themselves machine-generated (Appendix 1). The human
   reasonableness review (Appendix 4) judged 292 of 293 codes on a fresh 100-case random sample
   reasonable, with no timing code judged unreasonable, and upheld 32 of 33 late-claim codes
   across its two samples. Findings should be read as patterns in the published decision
   record, coded by a validated but imperfect model.

---
*Appendices: 1 — the benchmark sample · 2 — model validation · 3 — production run ·
4 — human review · 5 — findings (this notebook).*
